# Statistik — Job Market Indonesia
Dataset kecil & riil: uji dilakukan dengan sadar batasannya — semua kesimpulan disertai keterbatasan, bukan overclaim.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
lv = pd.read_csv('../data/processed/lowongan.csv')
sk = pd.read_csv('../data/processed/job_skill.csv')

## 1. Apakah judul 'analyst' berhubungan dengan senioritas?
Chi-square: `peran_analyst` × `senioritas`

In [2]:
ct = pd.crosstab(lv.peran_analyst, lv.senioritas)
chi2, p, dof, _ = stats.chi2_contingency(ct)
n_ = ct.values.sum()
v = np.sqrt(chi2 / (n_ * (min(ct.shape) - 1)))
print(ct)
print(f'chi2={chi2:.1f}, p={p:.4f}, dof={dof}, CramérV={v:.3f}')
print('p<0.05:', p < 0.05)

senioritas     Junior / Entry  Mid  Senior / Lead
peran_analyst                                    
False                       7   74             26
True                       14  257             87
chi2=1.3, p=0.5109, dof=2, CramérV=0.054
p<0.05: False


## 2. Apakah lowongan analyst lebih terbuka ke fresh grad?
Chi-square: `peran_analyst` × `fresh_grad_ok`

In [3]:
ct2 = pd.crosstab(lv.peran_analyst, lv.fresh_grad_ok)
chi2, p, dof, _ = stats.chi2_contingency(ct2)
rate = lv.groupby('peran_analyst').fresh_grad_ok.mean()
print(ct2, '\n', rate.round(3))
print(f'chi2={chi2:.2f}, p={p:.4f}')

fresh_grad_ok  False  True 
peran_analyst              
False             91     16
True             320     38 
 peran_analyst
False    0.150
True     0.106
Name: fresh_grad_ok, dtype: float64
chi2=1.12, p=0.2904


## 3. Gaji: Junior vs Senior
⚠️ Hanya ~17 lowongan menampilkan gaji → power sangat rendah. Hasil bersifat indikatif, BUKAN bukti.

In [4]:
bg = lv.dropna(subset=['gaji_max_jt'])
print(bg.groupby('senioritas').gaji_max_jt
        .agg(['count','median','mean']).round(1))
jun = bg[bg.senioritas=='Junior / Entry'].gaji_max_jt
sen = bg[bg.senioritas=='Senior / Lead'].gaji_max_jt
if len(jun) >= 3 and len(sen) >= 3:
    u, p = stats.mannwhitneyu(jun, sen, alternative='less')
    print(f'Mann-Whitney (junior < senior): U={u:.0f}, p={p:.4f}')
else:
    print('n per grup terlalu kecil untuk uji — dilaporkan deskriptif saja')

                count  median  mean
senioritas                         
Junior / Entry      2     6.8   6.8
Mid                14    10.0  10.9
Senior / Lead       2    22.5  22.5
n per grup terlalu kecil untuk uji — dilaporkan deskriptif saja


## 4. Skill × keluarga peran — mana yang membedakan?
Untuk tiap skill top-8: chi-square hadir/tidak × role_family (family kecil digabung 'Lainnya').

In [5]:
lv2 = lv.copy()
top_fam = lv2.role_family.value_counts()
lv2['fam'] = np.where(lv2.role_family.isin(top_fam[top_fam>=20].index),
                      lv2.role_family, 'Lainnya')
hasil = []
for s in sk.skill.value_counts().head(8).index:
    ids = set(sk[sk.skill == s].id_lowongan)
    lv2['ada'] = lv2.id_lowongan.isin(ids)
    ct = pd.crosstab(lv2.fam, lv2.ada)
    if ct.shape == (len(lv2.fam.unique()), 2):
        chi2, p, dof, _ = stats.chi2_contingency(ct)
        hasil.append((s, round(chi2,1), round(p,4)))
pd.DataFrame(hasil, columns=['skill','chi2','p']).sort_values('p')

,skill,chi2,p
0,SQL,48.3,0.0
1,Python,102.3,0.0
2,Excel,54.7,0.0
3,Visualisasi Data,80.1,0.0
4,Statistik,82.0,0.0
5,Power BI,41.4,0.0
6,ETL / Data Pipeline,155.8,0.0
7,Tableau,33.5,0.0


## Kesimpulan
- Uji signifikan → laporkan; tidak signifikan → katakan jujur.
- Dataset riil berukuran 295: efek kecil tidak akan terdeteksi secara statistik — itu sendiri temuan metodologis.
- Analisis gaji dibatasi ~6% coverage: kesimpulan gaji bersifat hipotesis, bukan estimasi pasar.